# [WIP] Proposed Redesign of SB-MOABB DataIO

In [5]:
%%capture
!pip install speechbrain moabb mne mne_bids

## Utilities

In [6]:
import functools
import warnings

import mne


def hide_mne_output(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        old_level = mne.set_log_level(verbose=False, return_old_level=True)
        with warnings.catch_warnings():
            warnings.filterwarnings(action="ignore", module="mne")

            res = func(*args, **kwargs)
        mne.set_log_level(old_level)

        return res

    return wrapper

# Extend DynamicItemDataset to support BIDS and MOABB datasets

In [7]:
import json
from pathlib import Path
from typing import Any

import mne
from mne_bids import BIDSPath, get_bids_path_from_fname, read_raw_bids
from moabb.datasets import BNCI2014_001
from moabb.datasets import download as dl
from moabb.datasets.base import BaseDataset as BaseMOABBDataset
from moabb.datasets.bids_interface import camel_to_kebab_case
from typing_extensions import Optional, Self

In [8]:
import speechbrain as sb
from speechbrain.dataio.dataset import DynamicItemDataset

In [9]:


class RawEEGDataset(DynamicItemDataset):

    def __init__(self, data, dynamic_items=[], output_keys=[]):
        # TODO set dynamic_item to load data
        dynamic_items = [self._load_raw] + dynamic_items
        super().__init__(
            data, dynamic_items=dynamic_items, output_keys=output_keys
        )

    @sb.utils.data_pipeline.takes("fpath")
    @sb.utils.data_pipeline.provides("raw", "info")
    @staticmethod
    def _load_raw(fpath):
        bids_path = get_bids_path_from_fname(fpath)
        raw = read_raw_bids(
            bids_path, extra_params=dict(preload=False), verbose=0
        )
        yield raw
        yield raw.info

    @classmethod
    def from_bids(
        cls,
        bids_path: Path | str | BIDSPath,
        json_path: str | Path,
        subjects=None,
        dynamic_items=[],
        output_keys=[],
    ) -> Self:
        """Creates a DynamicItemDataset from a BIDS EEG Dataset."""
        if not isinstance(bids_path, BIDSPath):
            bids_path = BIDSPath(root=bids_path)
        json_data = cls.load_or_create_json_data_from_bids(
            bids_path, json_path, subjects=subjects
        )

        return cls(
            data=json_data, dynamic_items=dynamic_items, output_keys=output_keys
        )

    @classmethod
    @hide_mne_output
    def from_moabb(
        cls,
        dataset: BaseMOABBDataset,
        json_path: str | Path,
        subjects=None,
        save_path: Optional[str] = None,
        dynamic_items=[],
        output_keys=[],
    ) -> Self:
        """Creates a DynamicItemDataset from a MOABB Dataset, by first
        converting it to BIDS format."""
        json_path = Path(json_path)
        if json_path.exists():
            with json_path.open() as fp:
                json_data = json.load(fp)

            return cls(
                data=json_data,
                dynamic_items=dynamic_items,
                output_keys=output_keys,
            )

        mne_path = Path(dl.get_dataset_path(dataset.code, save_path))
        cache_dir = f"MNE-BIDS-{camel_to_kebab_case(dataset.code)}"
        cache_path = mne_path / cache_dir

        for sub in subjects if subjects is not None else dataset.subject_list:
            dataset.get_data(
                subjects=[sub],
                cache_config=dict(use=True, save_raw=True, path=mne_path),
            )

        return cls.from_bids(
            bids_path=cache_path,
            json_path=json_path,
            dynamic_items=dynamic_items,
            output_keys=output_keys,
            subjects=subjects,
        )

    @classmethod
    def load_or_create_json_data_from_bids(
        cls,
        bids_path: BIDSPath,
        json_path: str | Path,
        subjects=None,
    ) -> dict[str, Any]:
        json_path = Path(json_path)
        if json_path.exists():
            with json_path.open() as fp:
                json_data = json.load(fp)
        else:
            json_data = cls.json_data_from_bids_path(bids_path)

            with json_path.open("w") as fp:
                json.dump(json_data, fp)

        if subjects is not None:
            json_data = {
                uid: data
                for uid, data in json_data.items()
                if data["subject"] in subjects
            }

        return json_data

    @classmethod
    def json_data_from_bids_path(cls, bids_path) -> dict[str, Any]:
        json_data = {}

        for path in bids_path.update(suffix="eeg").match(ignore_json=True):
            uid = path.fpath.name
            json_data[uid] = path.entities
            json_data[uid]["fpath"] = str(path.fpath)
        return json_data


class EpochedEEGDataset(RawEEGDataset):
    """Breaks the raw EEG signals up into epochs."""

    def __init__(self, data, dynamic_items=[], output_keys=[]):
        # TODO set dynamic_item to load data
        dynamic_items = [self._load_epoch] + dynamic_items
        super().__init__(
            data, dynamic_items=dynamic_items, output_keys=output_keys
        )

    @sb.utils.data_pipeline.takes("raw", "onset")
    @sb.utils.data_pipeline.provides("epoch", "times")
    @staticmethod
    def _load_epoch(raw, onset):
        yield raw.crop(tmin=onset / raw.info['sfreq'], verbose=False)

    @classmethod
    def json_data_from_bids_path(cls, bids_path) -> dict[str, Any]:
        raw_json_data = super().json_data_from_bids_path(bids_path)

        json_data = {}
        for uid, sample in raw_json_data.items():
            bids_path = get_bids_path_from_fname(sample["fpath"])
            raw = read_raw_bids(
                bids_path, extra_params=dict(preload=False), verbose=0
            )
            stim_channels = mne.utils._get_stim_channel(
                None, raw.info, raise_error=False
            )
            if len(stim_channels) > 0:
                # returns empty array if none found
                events = mne.find_events(raw, shortest_event=0, verbose=False)
                event_id = {}
            else:
                events, event_id = mne.events_from_annotations(
                    raw, verbose=False
                )

            event_id = {v: k for k, v in event_id.items()}

            for onset, _, event in events:
                label = event_id.get(event, int(event))
                event_sample = dict(**sample, label=label, onset=int(onset))
                event_uid = f"{uid}/{label}/{onset}"
                json_data[event_uid] = event_sample

        return json_data


# TEST IT OUT
dataset = EpochedEEGDataset.from_moabb(
    BNCI2014_001(),
    "data/MNE-BIDS-bnci2014-001-epoched.json",
    save_path="data",
    output_keys=[
        "label",
        "subject",
        "session",
        "epoch",
        "info",
    ],
)
len(dataset), dataset[0]

(5184,
 {'label': 'tongue',
  'subject': '1',
  'session': '0train',
  'epoch': <RawEDF | sub-1_ses-0train_task-imagery_run-0_desc-c6ddd98f4a171af69d0c8f3a1a73f5b5_eeg.edf, 22 x 96000 (384.0 s), ~36 KiB, data not loaded>,
  'info': <Info | 12 non-empty values
   bads: []
   ch_names: Fz, FC3, FC1, FCz, FC2, FC4, C5, C3, C1, Cz, C2, C4, C6, CP3, ...
   chs: 22 EEG
   custom_ref_applied: False
   description: Anonymized using a time shift to preserve age at acquisition
   dig: 25 items (3 Cardinal, 22 EEG)
   experimenter: mne_anonymize
   highpass: 0.0 Hz
   line_freq: 50.0
   lowpass: 125.0 Hz
   meas_date: 2025-01-28 22:50:36 UTC
   nchan: 22
   projs: []
   sfreq: 250.0 Hz
   subject_info: <subject_info | his_id: sub-1, sex: 0, hand: 0>
  >})